# **Preparing Dependancies**

In [ ]:
!pip install --upgrade diffusers transformers accelerate scipy safetensors torch torchvision

# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

model_id = 'runwayml/stable-diffusion-v1-5'
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()

## **Generate Image**

In [ ]:
def generate_simple_image(prompt, negative_prompt, seed):
    generator = torch.Generator('cuda').manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    ).images[0]
    return image

prompt = 'a broken satellite floating in space, highly detailed digital art, trending on artstation'
negative_prompt = 'photorealistic, realistic, photograph, 3d render, messy, blurry, low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration'
seed = 222

image_simple = generate_simple_image(prompt, negative_prompt, seed)
display(image_simple)

## **Generate Image with Hyperparameter Configuration**

In [ ]:
def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_steps):
    generator = torch.Generator('cuda').manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images[0]
    return image

image_advanced = generate_advanced_image(
    prompt=prompt, 
    negative_prompt=negative_prompt, 
    seed=seed, 
    guidance_scale=7.5, 
    num_inference_steps=50
)
display(image_advanced)

## **Guidance Scale Comparison**

In [ ]:
# Low Guidance Scale
img_low_gs = generate_advanced_image(prompt, negative_prompt, seed, guidance_scale=2.0, num_inference_steps=30)
print('Low Guidance Scale (2.0):')
display(img_low_gs)

# High Guidance Scale
img_high_gs = generate_advanced_image(prompt, negative_prompt, seed, guidance_scale=15.0, num_inference_steps=30)
print('High Guidance Scale (15.0):')
display(img_high_gs)

### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah:**   
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, kesesuaian dengan prompt, dan variasi visual yang terlihat."*

*   **Gambar dengan "Scale" Tinggi:**   
*"Jelaskan perbedaan yang terlihat dibandingkan guidance scale rendah, terutama pada detail gambar dan kedekatannya dengan prompt."*

## **Inference Steps Comparison**

In [ ]:
# Low Inference Steps (10)
img_low_steps = generate_advanced_image(prompt, negative_prompt, seed, guidance_scale=7.5, num_inference_steps=10)
print('Low Inference Steps (10):')
display(img_low_steps)

# High Inference Steps (50)
img_high_steps = generate_advanced_image(prompt, negative_prompt, seed, guidance_scale=7.5, num_inference_steps=50)
print('High Inference Steps (50):')
display(img_high_steps)

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**  
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, ketajaman, serta kemungkinan munculnya noise atau artefak."*
*   **Gambar dengan "Step" Tinggi:**  
*"Jelaskan perbedaan yang terlihat dibandingkan step rendah, terutama pada detail gambar, kehalusan hasil, dan stabilitas visual."*

## **Batch Inference from One Prompt**

In [ ]:
import matplotlib.pyplot as plt

def generate_batch_images(prompt, negative_prompt, seed, guidance_scale, num_inference_steps, num_images=4):
    generator = torch.Generator('cuda').manual_seed(seed)
    images = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[negative_prompt] * num_images,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for ax, img in zip(axes.flatten(), images):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

generate_batch_images(prompt, negative_prompt, seed, 7.5, 30)

## **Load Scheduler**

In [ ]:
from diffusers import EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler, DDIMScheduler

def load_scheduler(pipe, scheduler_name):
    if scheduler_name == 'Euler A':
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == 'DPM++':
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == 'DDIM':
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    return pipe

### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DPM++ Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DDIM Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*

# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Inpainting**

### **Load Model Inpainting**

In [ ]:
for sched in ['Euler A', 'DPM++', 'DDIM']:
    print(f'Generating with {sched}')
    pipe = load_scheduler(pipe, sched)
    img = generate_advanced_image(prompt, negative_prompt, seed, 7.5, 30)
    display(img)

### **Manual Masking**

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting', 
    torch_dtype=torch.float16
)
inpaint_pipe = inpaint_pipe.to('cuda')
inpaint_pipe.enable_attention_slicing()

# Prepare base image from previous task
base_image = image_advanced.copy().resize((512, 512))

### **Generate**

In [ ]:
from PIL import Image, ImageDraw

mask_image = Image.new('L', base_image.size, 0)
draw = ImageDraw.Draw(mask_image)
# Draw a mask block in the center
draw.rectangle((150, 150, 362, 362), fill=255)
display(mask_image)

## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

In [ ]:
def inpaint_engine(image, mask, prompt, seed=9):
    generator = torch.Generator('cuda').manual_seed(seed)
    result = inpaint_pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        generator=generator
    ).images[0]
    return result

inpaint_prompt = 'a detailed broken satellite part with sparking wires'
inpainted_image = inpaint_engine(base_image, mask_image, inpaint_prompt)
display(inpainted_image)

### **Masking with Segmentation Model**

In [ ]:
from transformers import pipeline
import numpy as np

segmenter = pipeline('image-segmentation', model='facebook/detr-resnet-50-panoptic')

### **Generate**

In [ ]:
def create_auto_mask(image):
    # Using a simple threshold for demonstration since panoptic might split too much
    # Let's just create a circular mask as a fallback if segmentation doesn't find object
    results = segmenter(image)
    if len(results) > 0:
        # Get the mask of the most prominent object
        mask = results[0]['mask']
        return mask
    else:
        m = Image.new('L', image.size, 0)
        ImageDraw.Draw(m).ellipse((100,100,400,400), fill=255)
        return m

auto_mask = create_auto_mask(base_image)
display(auto_mask)

## **Outpainting**

### **Prepare the Canvas**

In [ ]:
auto_inpainted_image = inpaint_engine(base_image, auto_mask, inpaint_prompt)
display(auto_inpainted_image)

### **Generate**

In [ ]:
def prepare_outpainting(image, direction='right', expand_pixels=256):
    width, height = image.size
    if direction == 'right':
        new_size = (width + expand_pixels, height)
        box, mask_box = (0, 0), (width, 0, width + expand_pixels, height)
    elif direction == 'left':
        new_size = (width + expand_pixels, height)
        box, mask_box = (expand_pixels, 0), (0, 0, expand_pixels, height)
    elif direction == 'bottom':
        new_size = (width, height + expand_pixels)
        box, mask_box = (0, 0), (0, height, width, height + expand_pixels)
    elif direction == 'top':
        new_size = (width, height + expand_pixels)
        box, mask_box = (0, expand_pixels), (0, 0, width, expand_pixels)
        
    new_image = Image.new('RGB', new_size, (255, 255, 255))
    new_image.paste(image, box)
    
    mask = Image.new('L', new_size, 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle(mask_box, fill=255)
    return new_image, mask

outpaint_base, outpaint_mask = prepare_outpainting(inpainted_image, 'right', 256)
display(outpaint_base)

## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

In [ ]:
outpaint_prompt = 'space background, stars, dark universe, nebula'
outpainted_result = inpaint_engine(outpaint_base, outpaint_mask, outpaint_prompt)
display(outpainted_result)

### **Generate**

In [ ]:
def prepare_zoom_out(image, expand_pixels=128):
    width, height = image.size
    new_size = (width + 2*expand_pixels, height + 2*expand_pixels)
    new_image = Image.new('RGB', new_size, (255, 255, 255))
    new_image.paste(image, (expand_pixels, expand_pixels))
    
    mask = Image.new('L', new_size, 255)
    draw = ImageDraw.Draw(mask)
    draw.rectangle((expand_pixels, expand_pixels, expand_pixels+width, expand_pixels+height), fill=0)
    return new_image, mask

zoom_base, zoom_mask = prepare_zoom_out(inpainted_image, 128)
display(zoom_base)

## **Base + Refiner Image Generation**

In [ ]:
zoom_result = inpaint_engine(zoom_base, zoom_mask, outpaint_prompt)
display(zoom_result)